In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
import casadi as ca
from IPython.display import clear_output, display
import matplotlib.ticker as ticker

def gaussian_kernel(X, Y, sigma=100):
    XX = np.sum(X ** 2, axis=0).reshape(-1, 1)
    YY = np.sum(Y ** 2, axis=0).reshape(1, -1)
    distances = XX + YY - 2 * np.dot(X.T, Y)
    return np.exp(-distances / (2 * sigma ** 2))

def gaussian_kernel_casadi(X1, X2, sigma=1.0):
    """
    Gaussian (RBF) kernel for CasADi symbolic variables.
    
    Args:
        X1: A CasADi MX matrix of shape (n_features, n_samples1)
        X2: A CasADi MX matrix of shape (n_features, n_samples2)
        sigma: The bandwidth parameter of the RBF kernel.
        
    Returns:
        A CasADi MX matrix of shape (n_samples1, n_samples2) representing the kernel matrix.
    """
    # Compute the squared L2 norm of the difference between each pair of points
    X1_sq = ca.sum1(X1**2).T  # Shape (n_samples1, 1)
    X2_sq = ca.sum1(X2**2).T  # Shape (n_samples2, 1)
    
    # Compute the squared Euclidean distance matrix
    sq_dists = X1_sq - 2 * (X1.T @ X2) + X2_sq.T
    
    # Compute the Gaussian kernel matrix
    K = ca.exp(-0.5 / sigma**2 * sq_dists)
    
    return K

In [ ]:
# perform the stabilisation with a pid controller to gather data

np.random.seed(4)

closed_loop = 1
show_animation = 0

# PID controller parameters
Kp_p = -3.2
Ki_p = -0.04
Kd_p = 0.5

Kp_c = -0.2
Ki_c = -0.0005
Kd_c = 0.01

integral_p = 0.0
integral_c = 0.0

if closed_loop:
    numSteps = 200
else:
    numSteps = 1000

u = np.random.normal(0, 1, numSteps)
y = np.zeros(numSteps)
x = np.zeros((4, numSteps))


reset_time = []
seed = 3

env = gym.make("InvertedPendulum-v4", render_mode="rgb_array")
 
obs, info = env.reset(seed=seed)

for k in range(numSteps):
    # Simple PID controller for angle stabilization
    obs += np.random.normal(0, 0.00001, (np.size(obs)))
    angle = obs[1]  # Angle of the pendulum
    dangle = obs[3]
    pos = obs[0]
    speed = obs[2]
    error_p = -angle  # Target is 0 angle
    integral_p += error_p
    error_c = -pos  # Target is 0 angle
    integral_c += error_c
    
    if closed_loop:
        u[k] = Kp_p * error_p + Ki_p * integral_p + Kd_p * dangle \
             + Kp_c * error_c + Ki_c * integral_c + Kd_c * speed + np.random.normal(0, 0.2)
        
    x[:,k] = obs
    
    # Clip the control input to the action space limits
    u[k] = np.clip(u[k], env.action_space.low[0], env.action_space.high[0])
    obs, reward, done, truncated, info = env.step([u[k]])
 

    if done or truncated or np.abs(angle) > 0.3:
        seed += 1
        obs, info = env.reset(seed = seed)
        integral_p = 0.0
        integral_c = 0.0
        reset_time += [k]
        
env.close()
reset_time += [k]
y = x

In [ ]:
plt.ion()  # Turn on interactive mode

# Create four figures
fig1, ax1 = plt.subplots(figsize=(4, 3))
plt.xlabel('Time step')
plt.subplots_adjust(left=0.2, bottom=0.2, right=0.9, top=0.9) 
fig2, ax2 = plt.subplots(figsize=(4, 3))
plt.xlabel('Time step')
plt.gca().yaxis.set_major_locator(ticker.MultipleLocator(base=np.pi / 2))
plt.gca().yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda val, pos: (
        "0" if val == 0 else
        "π" if np.isclose(val, np.pi) else
        "2π" if np.isclose(val, 2 * np.pi) else
        "π/2" if np.isclose(val, np.pi / 2) else
        "3π/2" if np.isclose(val, 3 * np.pi / 2) else
        f"{val / np.pi:.0g}π"
    ))
)
plt.subplots_adjust(left=0.2, bottom=0.2, right=0.9, top=0.9) 
fig3, ax3 = plt.subplots(figsize=(4, 3))
plt.xlabel('Time step')
plt.subplots_adjust(left=0.2, bottom=0.2, right=0.9, top=0.9) 
fig4, ax4 = plt.subplots(figsize=(4, 3))
plt.xlabel('Time step')
plt.subplots_adjust(left=0.2, bottom=0.2, right=0.9, top=0.9) 

In [ ]:
# Perform the grid search of the Kernel DeePC 

T_ini = 1
T_sim = 150

N_o = 20

# the grid of parameters to search
gamma_values = [1e-6, 1e-5, 1e-4]
sigma_values = [0.01, 0.05, 0.1, 0.2, 0.5, 1, 1.5, 2, 5, 10, 20, 50]

# Initialize storage for results
results = {}

# Color map to generate unique colors
colors = plt.cm.viridis(np.linspace(0, 1, len(gamma_values) * len(sigma_values)))
color_idx = 0

L_o = T_ini + N_o  # Hankel matrix row
u_ini = np.zeros((T_ini,))
y_ini = np.zeros((T_ini*4,))

for gamma in gamma_values:
    for sigma_o in sigma_values:
        results[(gamma, sigma_o)] = 0
        U_p_o = []
        U_f_o = []
        Y_p_o = []
        Y_f_o = []

        last_t = 0
        for t in reset_time:
            y_section = y[:, last_t:t+1]
            u_section = u[last_t:t+1]
            last_t = t
            
            numSteps_section = len(u_section)
            if numSteps_section - L_o > 0:
                H_u_o = np.zeros((L_o, numSteps_section - L_o + 1))
                H_y_o = np.zeros((L_o * 4, numSteps_section - L_o + 1))
                for i in range(L_o):
                    H_u_o[i, :] = u_section[i:i + numSteps_section - L_o + 1]
                    H_y_o[i*4:(i+1)*4, :] = y_section[:, i:i + numSteps_section - L_o + 1]
                
                U_p_o += [H_u_o[:T_ini, :]]
                U_f_o += [H_u_o[T_ini:, :]]
                
                Y_p_o += [H_y_o[:T_ini*4, :]]
                Y_f_o += [H_y_o[T_ini*4:, :]]
            
        U_p_o = np.hstack(U_p_o)
        U_f_o = np.hstack(U_f_o)
        Y_p_o = np.hstack(Y_p_o)
        Y_f_o = np.hstack(Y_f_o)
            
        H_o = np.vstack((U_p_o, Y_p_o, U_f_o, Y_f_o))
        D_o = np.vstack((U_p_o, Y_p_o, U_f_o))
            
            # Gram matrix
        K_o = gaussian_kernel(D_o, D_o, sigma_o)

        H_c_o = len(H_o[0])

        # RoKDeePC (open-loop)
        u_o = np.zeros((T_sim,))


        env = gym.make("InvertedPendulum", render_mode="rgb_array")

        obs, info = env.reset(seed=3)
        lb = env.action_space.low[0]
        ub = env.action_space.high[0]

        y1 = obs
        y_ini = obs


        for i in range(T_sim):
            
            opti = ca.Opti()
            x_o = opti.variable(N_o + 5 * T_ini)
            g_o = opti.variable(H_c_o)
            objective = ca.sum1(x_o[5*T_ini:] ** 2)
            objective += ca.mtimes([(Y_f_o @ g_o).T, np.kron(np.eye(N_o),np.diag([10, 1, 0.1, 0.1])), (Y_f_o @ g_o)])
            constraints = []
            gaussian_kernel_ = gaussian_kernel_casadi(D_o, x_o, sigma_o)
            constraints.append((K_o + gamma * ca.MX.eye(H_c_o)) @ g_o == gaussian_kernel_)
            constraints.append(x_o[5*T_ini:] >= lb)
            constraints.append(x_o[5*T_ini:] <= ub)
            constraints.append(x_o[:T_ini] == u_ini)
            constraints.append(x_o[T_ini:5*T_ini] == y_ini)
            
            for constraint in constraints:
                opti.subject_to(constraint)

            # Set the objective in Opti
            opti.minimize(objective)
            
            # Set initial guesses for the variables
            initial_guess = np.zeros(N_o + H_c_o + 5 * T_ini)
            opti.set_initial(ca.vertcat(x_o, g_o), initial_guess)
            
            # Solver options
            opts = {'ipopt.print_level': 0, 'print_time': 0}
            opti.solver('ipopt', opts)
            
            # Solve the problem
            sol = opti.solve()
            
            # Extract the solution
            u_o_opt = sol.value(x_o[5 * T_ini:N_o + 5 * T_ini])
            g_o_opt = sol.value(g_o)

            
            obs, reward, done, truncated, info = env.step([u_ini[-1]])
            
            u1_o = u_o_opt[0]
            u_o[i] = u1_o
            y1_old = y1
            y1 = obs
            
            u_ini = np.hstack((u_ini, u1_o))[1:]
            y_ini = np.hstack((y_ini, y1))[4:]
            
            results[(gamma, sigma_o)] += y1.T @ np.diag([10, 1, 0.1, 0.1]) @ y1
                            
            ax1.plot([i, i+1], [y1_old[0], y1[0]], '-', linewidth=1.5, color = colors[color_idx])
            
            ax2.plot([i, i+1], [y1_old[1], y1[1]], '-', linewidth=1.5, color = colors[color_idx])
            
            ax3.plot([i, i+1], [y1_old[2], y1[2]], '-', linewidth=1.5, color = colors[color_idx])
            
            ax4.plot([i, i+1], [y1_old[3], y1[3]], '-', linewidth=1.5, color = colors[color_idx])
            
            # Redraw the updated plots
            fig1.canvas.draw()
            fig1.canvas.flush_events()
            fig2.canvas.draw()
            fig2.canvas.flush_events()
            fig3.canvas.draw()
            fig3.canvas.flush_events()
            fig4.canvas.draw()
            fig4.canvas.flush_events()
            clear_output(wait=True)
            display(fig1)
            display(fig2)
            display(fig3)
            display(fig4)
            plt.pause(0.001)
        color_idx += 1
            
        env.close()

        plt.ioff()
        plt.show()


In [ ]:
fig1.savefig('plots/stabilise/gs position.pdf')
fig2.savefig('plots/stabilise/gs angle.pdf')
fig3.savefig('plots/stabilise/gs linear velocity.pdf')
fig4.savefig('plots/stabilise\gs angular velocity.pdf')